In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
diabetes = pd.read_csv('data/diabetes_data.csv')
diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Gender
0,6,98,58,33,190,34.0,0.430,43,0,Female
1,2,112,75,32,0,35.7,0.148,21,0,Female
2,2,108,64,0,0,30.8,0.158,21,0,Female
3,8,107,80,0,0,24.6,0.856,34,0,Female
4,7,136,90,0,0,29.9,0.210,50,0,Female


In [14]:
diabetes_df = diabetes.copy()



In [15]:
print("=== АНАЛИЗ ДУБЛИКАТОВ ===")
print(f"Исходный размер данных: {diabetes_df.shape}")

# Подсчет дубликатов
duplicate_count = diabetes_df.duplicated().sum()
print(f"Количество полных дубликатов: {duplicate_count}")

if duplicate_count > 0:
    # Показать дубликаты
    duplicates = diabetes_df[diabetes_df.duplicated(keep=False)]
    print("\nВсе дубликаты:")
    print(duplicates.sort_values(by=diabetes_df.columns.tolist()))
    
    # Удаление дубликатов
    diabetes_df_cleaned = diabetes_df.drop_duplicates()
    
    print(f"\nРазмер после удаления дубликатов: {diabetes_df_cleaned.shape}")
    print(f"Удалено {duplicate_count} строк(и)")
    
    # Сохранение результата
    diabetes_df_cleaned.to_csv('diabetes_df_cleaned.txt', index=False)
    print("Очищенные данные сохранены в 'diabetes_df_cleaned.txt'")
else:
    print("Дубликаты не найдены!")

=== АНАЛИЗ ДУБЛИКАТОВ ===
Исходный размер данных: (778, 10)
Количество полных дубликатов: 10

Все дубликаты:
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
7              0      117              0              0        0  33.8   
775            0      117              0              0        0  33.8   
6              1       71             48             18       76  20.4   
774            1       71             48             18       76  20.4   
2              2      108             64              0        0  30.8   
770            2      108             64              0        0  30.8   
1              2      112             75             32        0  35.7   
769            2      112             75             32        0  35.7   
8              4      154             72             29      126  31.3   
776            4      154             72             29      126  31.3   
9              5      147             78              0        0  33.7   
777

In [16]:
print("=== ОБРАБОТКА ПРОПУСКОВ ===")

# Столбцы, где 0 означает пропуск (медицинские показатели)
medical_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Статистика до замены
print("До замены:")
for col in medical_columns:
    zeros = (diabetes_df_cleaned[col] == 0).sum()
    print(f"  {col}: {zeros} нулей")

# Замена 0 на NaN
diabetes_df_cleaned[medical_columns] = diabetes_df_cleaned[medical_columns].replace(0, np.nan)

# Статистика после замены
print("\nПосле замены:")
print("Количество пропусков (NaN):")
print(diabetes_df_cleaned[medical_columns].isnull().sum())

# Общая информация
print(f"\nВсего пропусков: {diabetes_df_cleaned[medical_columns].isnull().sum().sum()}")


=== ОБРАБОТКА ПРОПУСКОВ ===
До замены:
  Glucose: 5 нулей
  BloodPressure: 35 нулей
  SkinThickness: 227 нулей
  Insulin: 374 нулей
  BMI: 11 нулей

После замены:
Количество пропусков (NaN):
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

Всего пропусков: 652


C:\Users\Сергей\AppData\Local\Temp\ipykernel_19576\1487247912.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diabetes_df_cleaned[medical_columns] = diabetes_df_cleaned[medical_columns].replace(0, np.nan)


In [17]:
# Предполагаем, что diabetes_df_cleaned уже загружен
print("=== ОБРАБОТКА ПРОПУСКОВ В diabetes_df_cleaned ===")
print(f"Исходный размер данных: {diabetes_df_cleaned.shape}")

# 1. Замена 0 на NaN в медицинских показателях
medical_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("\n1. Замена 0 на NaN в медицинских показателях:")
for col in medical_columns:
    zeros_before = (diabetes_df_cleaned[col] == 0).sum()
    diabetes_df_cleaned[col] = diabetes_df_cleaned[col].replace(0, np.nan)
    zeros_after = diabetes_df_cleaned[col].isnull().sum()
    print(f"  {col}: {zeros_before} нулей → {zeros_after} пропусков")

# 2. Удаление признаков с долей пропусков > 30%
print("\n2. Удаление признаков с пропусками > 30%:")

# Расчет доли пропусков
missing_ratio = (diabetes_df_cleaned.isnull().sum() / len(diabetes_df_cleaned)) * 100

print("\nДоля пропусков по столбцам (%):")
for column, ratio in missing_ratio.items():
    if ratio > 0:  # Показываем только столбцы с пропусками
        print(f"  {column}: {ratio:.2f}%")

# Определение столбцов для удаления
columns_to_drop = missing_ratio[missing_ratio > 30].index.tolist()

if columns_to_drop:
    print(f"\nУдаляем столбцы: {columns_to_drop}")
    diabetes_df_cleaned = diabetes_df_cleaned.drop(columns=columns_to_drop)
    print(f"Размер после удаления: {diabetes_df_cleaned.shape}")
else:
    print("Нет столбцов с долей пропусков > 30%")

# 3. Итоговая информация
print(f"\n=== РЕЗУЛЬТАТ ===")
print(f"Финальный размер данных: {diabetes_df_cleaned.shape}")
print(f"Оставшиеся столбцы: {list(diabetes_df_cleaned.columns)}")

# Количество пропусков в оставшихся столбцах
remaining_missing = diabetes_df_cleaned.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) > 0:
    print("\nПропуски в оставшихся столбцах:")
    for col, count in remaining_missing.items():
        ratio = (count / len(diabetes_df_cleaned)) * 100
        print(f"  {col}: {count} пропусков ({ratio:.2f}%)")
else:
    print("\nВсе пропуски обработаны")

=== ОБРАБОТКА ПРОПУСКОВ В diabetes_df_cleaned ===
Исходный размер данных: (768, 10)

1. Замена 0 на NaN в медицинских показателях:
  Glucose: 0 нулей → 5 пропусков
  BloodPressure: 0 нулей → 35 пропусков
  SkinThickness: 0 нулей → 227 пропусков
  Insulin: 0 нулей → 374 пропусков
  BMI: 0 нулей → 11 пропусков

2. Удаление признаков с пропусками > 30%:

Доля пропусков по столбцам (%):
  Glucose: 0.65%
  BloodPressure: 4.56%
  SkinThickness: 29.56%
  Insulin: 48.70%
  BMI: 1.43%

Удаляем столбцы: ['Insulin']
Размер после удаления: (768, 9)

=== РЕЗУЛЬТАТ ===
Финальный размер данных: (768, 9)
Оставшиеся столбцы: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'Gender']

Пропуски в оставшихся столбцах:
  Glucose: 5 пропусков (0.65%)
  BloodPressure: 35 пропусков (4.56%)
  SkinThickness: 227 пропусков (29.56%)
  BMI: 11 пропусков (1.43%)


C:\Users\Сергей\AppData\Local\Temp\ipykernel_19576\2009869319.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  diabetes_df_cleaned[col] = diabetes_df_cleaned[col].replace(0, np.nan)


In [18]:
# Удаление строк с более чем 2 пропусками
diabetes_df_cleaned = diabetes_df_cleaned[diabetes_df_cleaned.isnull().sum(axis=1) <= 2]

print(f"Результат: {diabetes_df_cleaned.shape}")

Результат: (761, 9)


In [19]:
print("=== ЗАМЕНА ПРОПУСКОВ НА МЕДИАНУ ===")
print(f"Размер данных перед обработкой: {diabetes_df_cleaned.shape}")

# 1. Найдем столбцы с пропусками
columns_with_missing = diabetes_df_cleaned.columns[diabetes_df_cleaned.isnull().any()].tolist()
print(f"Столбцы с пропусками: {columns_with_missing}")

# 2. Вычислим медиану для каждого столбца с пропусками
medians = {}
for column in columns_with_missing:
    median_value = diabetes_df_cleaned[column].median()
    medians[column] = median_value
    missing_count = diabetes_df_cleaned[column].isnull().sum()
    print(f"{column}: медиана = {median_value:.2f}, пропусков = {missing_count}")

# 3. Особенно для SkinThickness
skin_thickness_median = diabetes_df_cleaned['SkinThickness'].median()
print(f"\nМедиана SkinThickness: {skin_thickness_median:.2f}")

# 4. Дополнительная статистика по SkinThickness
print(f"\nДетальная статистика SkinThickness:")
print(f"  Медиана: {skin_thickness_median:.2f}")
print(f"  Среднее: {diabetes_df_cleaned['SkinThickness'].mean():.2f}")
print(f"  Минимум: {diabetes_df_cleaned['SkinThickness'].min():.2f}")
print(f"  Максимум: {diabetes_df_cleaned['SkinThickness'].max():.2f}")
print(f"  Стандартное отклонение: {diabetes_df_cleaned['SkinThickness'].std():.2f}")
print(f"  Количество пропусков: {diabetes_df_cleaned['SkinThickness'].isnull().sum()}")

# 5. Замена пропусков на медиану во всех столбцах
for column in columns_with_missing:
    missing_before = diabetes_df_cleaned[column].isnull().sum()
    diabetes_df_cleaned[column] = diabetes_df_cleaned[column].fillna(medians[column])
    missing_after = diabetes_df_cleaned[column].isnull().sum()
    print(f"Заполнено пропусков в {column}: {missing_before} → {missing_after}")

# 6. Проверка результата
print(f"\n=== РЕЗУЛЬТАТ ===")
print(f"Размер данных: {diabetes_df_cleaned.shape}")
print(f"Общее количество пропусков после заполнения: {diabetes_df_cleaned.isnull().sum().sum()}")

# 7. Проверка распределения после заполнения
print(f"\nСтатистика SkinThickness после заполнения:")
print(f"  Медиана: {diabetes_df_cleaned['SkinThickness'].median():.2f}")
print(f"  Среднее: {diabetes_df_cleaned['SkinThickness'].mean():.2f}")
print(f"  Минимум: {diabetes_df_cleaned['SkinThickness'].min():.2f}")
print(f"  Максимум: {diabetes_df_cleaned['SkinThickness'].max():.2f}")

=== ЗАМЕНА ПРОПУСКОВ НА МЕДИАНУ ===
Размер данных перед обработкой: (761, 9)
Столбцы с пропусками: ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']
Glucose: медиана = 117.00, пропусков = 5
BloodPressure: медиана = 72.00, пропусков = 28
SkinThickness: медиана = 29.00, пропусков = 220
BMI: медиана = 32.30, пропусков = 4

Медиана SkinThickness: 29.00

Детальная статистика SkinThickness:
  Медиана: 29.00
  Среднее: 29.15
  Минимум: 7.00
  Максимум: 99.00
  Стандартное отклонение: 10.48
  Количество пропусков: 220
Заполнено пропусков в Glucose: 5 → 0
Заполнено пропусков в BloodPressure: 28 → 0
Заполнено пропусков в SkinThickness: 220 → 0
Заполнено пропусков в BMI: 4 → 0

=== РЕЗУЛЬТАТ ===
Размер данных: (761, 9)
Общее количество пропусков после заполнения: 0

Статистика SkinThickness после заполнения:
  Медиана: 29.00
  Среднее: 29.11
  Минимум: 7.00
  Максимум: 99.00


In [20]:
def find_iqr_outliers(data, column):
    """Находит выбросы методом IQR для указанного столбца"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    
    return outliers, lower, upper

# Поиск выбросов
outliers, lower_bound, upper_bound = find_iqr_outliers(diabetes_df_cleaned, 'SkinThickness')

print(f"=== ВЫБРОСЫ В SkinThickness ===")
print(f"Границы: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Найдено выбросов: {len(outliers)}")
print(f"Процент выбросов: {(len(outliers) / len(diabetes_df_cleaned) * 100):.2f}%")

if len(outliers) > 0:
    print(f"\nСтатистика выбросов:")
    print(f"  Min: {outliers['SkinThickness'].min():.2f}")
    print(f"  Max: {outliers['SkinThickness'].max():.2f}")
    print(f"  Median: {outliers['SkinThickness'].median():.2f}")

=== ВЫБРОСЫ В SkinThickness ===
Границы: [14.50, 42.50]
Найдено выбросов: 87
Процент выбросов: 11.43%

Статистика выбросов:
  Min: 7.00
  Max: 99.00
  Median: 43.00
